### RAG pipelines- Data Ingestion to vector DB pipeline

In [3]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [9]:
from pathlib import Path
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents
# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data/pdf_files")

Found 2 PDF files to process

Processing: combinatorial.pdf
  ✓ Loaded 2 pages

Processing: ethics.pdf
  ✓ Loaded 2 pages

Total documents loaded: 4


In [11]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft Reporting Services PDF Rendering Extension 2022.1.0.0', 'creator': 'Microsoft Reporting Services 2022.1.0.0', 'creationdate': '2026-06-07T12:47:24+05:30', 'title': 'rdlCourseSyllabusNew', 'author': '', 'subject': '', 'source': '..\\data\\pdf_files\\combinatorial.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'combinatorial.pdf', 'file_type': 'pdf'}, page_content='CSE357\n:\nCOMBINATORIAL STUDIES\nCourse Outcomes:\nCO1 :: \nunderstand the fundamental computer science concepts, including data structures,  \nalgorithms, databases, operating systems, and computer networks, essential for technical  \ninterviews.\nCO2 :: \nassess the problem-solving skills specific to coding challenges and algorithmic problems  \nfrequently encountered in technical interviews.\nCO3 :: \narticulate a comprehensive command of Object-Oriented Programming principles,  \nenhancing readiness to excel in technical interviews.\nUnit I\nOperating System

In [13]:
### text splitting get into chunks
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into chunks"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, 
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.create_documents([doc.page_content for doc in documents], metadatas=[doc.metadata for doc in documents])
    print(f"Split into {len(split_docs)} chunks")
    if split_docs:
        print(f"Example chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    return split_docs

In [16]:
chunks=split_documents(all_pdf_documents)
chunks

Split into 10 chunks
Example chunk:
Content: CSE357
:
COMBINATORIAL STUDIES
Course Outcomes:
CO1 :: 
understand the fundamental computer science concepts, including data structures,  
algorithms, databases, operating systems, and computer networ...
Metadata: {'producer': 'Microsoft Reporting Services PDF Rendering Extension 2022.1.0.0', 'creator': 'Microsoft Reporting Services 2022.1.0.0', 'creationdate': '2026-06-07T12:47:24+05:30', 'title': 'rdlCourseSyllabusNew', 'author': '', 'subject': '', 'source': '..\\data\\pdf_files\\combinatorial.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'combinatorial.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft Reporting Services PDF Rendering Extension 2022.1.0.0', 'creator': 'Microsoft Reporting Services 2022.1.0.0', 'creationdate': '2026-06-07T12:47:24+05:30', 'title': 'rdlCourseSyllabusNew', 'author': '', 'subject': '', 'source': '..\\data\\pdf_files\\combinatorial.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'combinatorial.pdf', 'file_type': 'pdf'}, page_content='CSE357\n:\nCOMBINATORIAL STUDIES\nCourse Outcomes:\nCO1 :: \nunderstand the fundamental computer science concepts, including data structures,  \nalgorithms, databases, operating systems, and computer networks, essential for technical  \ninterviews.\nCO2 :: \nassess the problem-solving skills specific to coding challenges and algorithmic problems  \nfrequently encountered in technical interviews.\nCO3 :: \narticulate a comprehensive command of Object-Oriented Programming principles,  \nenhancing readiness to excel in technical interviews.\nUnit I\nOperating System

### emabeding and vectorstore DB

In [17]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity


c:\Users\chand\OneDrive\Desktop\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [18]:
class embeddingManager:
    """handle document embedding generation using sentence transformers
    """
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initialize the embedding manager
        Args:
            model_name (str, optional): The name of the sentence transformer model to use. Defaults to "all-MiniLM-L6-v2".
        """
        self.model_name = model_name
        self.model=None
        self._load_model()
    
    def _load_model(self):
        """Load the sentence transformer model
        """
        try:
            print(f"Loading model: {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully.:{self.model.get_sentence_embedding_dimension()} dimensions")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts
        Args:
            texts (List[str]): A list of strings to generate embeddings for
        Returns:
            np.ndarray: A 2D array of shape (len(texts), embedding_dimension) containing the embeddings
        """
        if not self.model:
            raise ValueError("Model not loaded. Call _load_model() first.")
     
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
      
## initialize embedding manager
embedding_manager = embeddingManager()
embedding_manager


Loading model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3169.30it/s]


Model loaded successfully.:384 dimensions


C:\Users\chand\AppData\Local\Temp\ipykernel_12252\2946198722.py:19: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully.:{self.model.get_sentence_embedding_dimension()} dimensions")


### vector store

In [20]:
class VectorStore:
    """
    A simple vector store using ChromaDB to store document embeddings and metadata.
    """

    def __init__(
        self,
        collection_name: str = "pdf_chunks",
        persist_directory: str = "../data/vector_store",
    ):
        """
        Initialize the vector store.

        Args:
            collection_name: Name of the ChromaDB collection.
            persist_directory: Directory where ChromaDB data is stored.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None

        self._initialize_store()

    def _initialize_store(self):
        """
        Initialize the ChromaDB client and collection.
        """
        try:
            # Create persistence directory
            os.makedirs(self.persist_directory, exist_ok=True)

            # Create persistent ChromaDB client
            self.client = chromadb.PersistentClient(
                path=self.persist_directory
            )

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"hnsw:space": "cosine"},
            )

            print(
                f"Vector store initialized with collection: "
                f"{self.collection_name}"
            )
            print(
                f"Current number of documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:
            print(f"Error initializing ChromaDB: {e}")
            raise

    def add_documents(
        self,
        documents: List[Any],
        embeddings: np.ndarray,
    ):
        """
        Add documents and embeddings to ChromaDB.

        Args:
            documents: List of LangChain documents.
            embeddings: Numpy array of embeddings.
        """
        if len(documents) != len(embeddings):
            raise ValueError(
                "Number of documents must match number of embeddings"
            )

        print(f"Adding {len(documents)} documents to vector store...")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(
            zip(documents, embeddings)
        ):
            # Unique document ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # ChromaDB metadata must contain primitive values
            metadata = {
                k: (
                    v
                    if isinstance(v, (str, int, float, bool))
                    else str(v)
                )
                for k, v in doc.metadata.items()
            }

            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)

            metadatas.append(metadata)

            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text,
            )

            print(
                f"Successfully added {len(documents)} documents"
            )
            print(
                f"Total documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise


# Create vector store
vectorstore = VectorStore()
print(vectorstore)

Vector store initialized with collection: pdf_chunks
Current number of documents in collection: 0
